# Factors that lead to BS3/PS3 low performance.

In this case study, we will take a look at BS3/PS3. Specifically, we will try to find the root cause of low accuracy in BS3/PS3.

---



In [1]:
from sklearn.metrics import confusion_matrix, accuracy_score
from pathlib import Path
import sys
import json
import pandas as pd

APPLIES_LABELS = [False, True]

# ----------------- LOAD BS3 SAMPLE ----------------- #
project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from evaluation.question1 import load_gold_answer_df
from pipeline import _safe_variant_name

gold_csv = project_root / "full_evaluation_dataset.csv"

gold_df = load_gold_answer_df(str(gold_csv))

predictions = []
prediction_dir = project_root / "outputs/batch_20260708_132813"

if not prediction_dir.exists():
    raise FileNotFoundError(
        f"Could not found predictions at {prediction_dir}"
    )

non_empty_count = 0
predictions = []

for file in sorted(prediction_dir.glob("*.json")):
    if file.name == "batch_summary.json":
        continue

    with open(file, "r", encoding="utf-8") as f:
        payload = json.load(f)

    variant = payload.get("variant")
    results = payload.get("results")

    has_bs3 = any(
        result.get("criterion") == "BS3" for result in results 
    )

    if not has_bs3:
        continue

    if not variant:
        continue
    variant = _safe_variant_name(variant)

    if not gold_df.loc[variant, "gold_applied"]:
        continue

    gold_answer = "BS3" in gold_df.loc[variant, "gold_applied"]

    bs3_prediction = next(
        (result for result in results if result.get("criterion") == "BS3"),
        None
    )

    if not bs3_prediction:
        continue

    pred_answer = bs3_prediction.get("applies")

    if not isinstance(pred_answer, bool):
        continue

    prediction = {
        "variant": variant,
        "criterion": "BS3",
        "evidence": bs3_prediction.get("evidence", "unknown"),
        "reasoning": bs3_prediction.get("reasoning", "unknown"),
        "tool_used": bs3_prediction.get("tool_used", "unknown"),
        "prediction": pred_answer,
        "gold_answer": gold_answer
    }

    # predictions.append(prediction)

    if prediction["prediction"] != prediction["gold_answer"]:
        predictions.append(prediction)

# ----------------- COMPUTE CONFUSION MATRIX ----------------- #
pred_df = pd.DataFrame(predictions)

pred_df = pred_df.to_string()
print(f"{"-" * 60}")
print("predictions: ")
print(pred_df)
print(f"{"-" * 60}")

------------------------------------------------------------
predictions: 
                    variant criterion                                                                                                     evidence                                                                                                                                                                                                              reasoning            tool_used  prediction  gold_answer
0     NM_000020_3_c_1445C_T       BS3               No functional experiments were found in the retrieved tool output (experiments list is empty).          According to the classification rules, Shortcut B applies: if there are no experiments, BS3 does not apply. Since the experiments list is empty, we cannot evaluate any functional assays for benign support.  functional_evidence       False         True
1      NM_000020_3_c_484C_T       BS3      No functional experiments were found in the retrieved tool output 

### Analysis of BS3 False-Negative Predictions

The table above shows five variants for which the framework produced false-negative BS3 predictions.

In every case:

* <u>The framework predicted that BS3 did not apply.</u>
* <u>The gold-standard classification indicated that BS3 should apply.</u>

---

#### Common Pattern

A consistent pattern appears across all five variants: the `functional_evidence` tool returned an empty `experiments` list.

Because no functional experiments were provided to the decision-making component, the framework followed the predefined shortcut rule:

> **If no functional experiments are found, BS3 does not apply.**
>
> The final BS3 decisions were therefore logically consistent with the evidence available to the model. However, the evidence itself was incomplete.

This suggests that the false-negative predictions were more likely caused by weaknesses in the upstream evidence-retrieval workflow than by problems with the BS3 classification instructions.

---

#### Possible Failure Points

There are at least two possible failure points in the current workflow.

##### 1. Failure to Retrieve Relevant PMIDs

The framework may fail to identify all publications that mention or experimentally evaluate a given variant.

If the relevant PMIDs are not retrieved, the downstream components never receive the papers containing the functional evidence.

The resulting workflow is:

```text
Relevant publication not retrieved
                ↓
Functional experiment not available to the framework
                ↓
Empty experiments list
                ↓
BS3 predicted as not applicable
```

##### 2. Failure to Extract Functional Experiments

Even when the correct PMIDs are successfully retrieved, the framework may not reliably identify every relevant functional experiment described in the publications.

Functional evidence may appear in:

* The main text
* Figures or figure captions
* Tables
* Supplementary materials
* Indirect or complex experimental descriptions

These sources may be difficult for the current extraction pipeline to recognize and process correctly.

---

### Interpretation of the Empty Experiments Lists

An empty `experiments` list does not necessarily mean that no functional evidence exists.

Instead, it may indicate one of the following:

1. The framework failed to retrieve the relevant publications.
2. The framework retrieved the publications but failed to extract the relevant functional experiments.

---


In [2]:
from tools.functional_evidence import analyze_variant
from pprint import pprint

variants = [
    "NM_000020_3_c_1445C_T", 
    "NM_000020_3_c_484C_T", 
    "NM_001114753_3_c_1316A_C", 
    "NM_001114753_3_c_1510G_A", 
    "NM_001114753_3_c_1844C_T",
]

results = {}
for variant in variants:
    normalized_variant = variant.strip()
    try:
        result = analyze_variant(normalized_variant)
        results[variant] = result
    except Exception as e:
        results[variant] = {
            "status": "error",
            "error": str(e)
        }
        
pprint(results)



No RSID available for the variant NM_000020_3_c_1445C_T
No RSID available for the variant NM_000020_3_c_484C_T
No RSID available for the variant NM_001114753_3_c_1316A_C
No RSID available for the variant NM_001114753_3_c_1510G_A
No RSID available for the variant NM_001114753_3_c_1844C_T
{'NM_000020_3_c_1445C_T': {'experiments': []},
 'NM_000020_3_c_484C_T': {'experiments': []},
 'NM_001114753_3_c_1316A_C': {'experiments': []},
 'NM_001114753_3_c_1510G_A': {'experiments': []},
 'NM_001114753_3_c_1844C_T': {'experiments': []}}


### Diagnosis of the BS3 False-Negative Results

By running the diagnostic code above, we found that the underlying problem was the absence of an available `rsID` for these variants.

The first step of the current `functional_evidence` workflow uses LitVar2 to retrieve PMIDs for publications related to a variant. However, this LitVar2 query requires an `rsID`. When no `rsID` is available, the workflow cannot retrieve any PMIDs and therefore fails at the very first stage.

> This explains why the tool returned an empty `experiments` list for these variants. The problem was not necessarily that no functional experiments existed. Instead, the workflow was unable to begin the publication-retrieval process because the required `rsID` was missing.

---

### Implemented Fallback Solution

I implemented two additional fallback methods for retrieving relevant publications when an `rsID` is unavailable. These fallback methods allow the framework to search for papers using alternative variant representations rather than relying exclusively on LitVar2 and `rsID`s.

However, I temporarily commented out these fallback methods after observing a major performance issue. The fallback searches can retrieve a very large number of publications, and the current `functional_evidence` tool attempts to examine too many of them. As a result, processing a single variant can take more than 10 minutes.

> Therefore, although the fallback methods improve publication-retrieval coverage, the current implementation is too slow for practical batch evaluation.

---

### Possible Approaches for Improving Performance

#### 1. Use LangGraph to Process Papers in Parallel

The paper-processing tasks could be distributed across multiple LangGraph nodes so that several publications are analyzed concurrently rather than sequentially.

#### 2. Use a Faster Model

The current model could be replaced with a lower-latency model optimized for high-throughput or near-instant inference. Because functional-evidence screening may require processing many papers, reducing the inference time per paper could substantially improve the overall runtime.

#### 3. Limit the Number of Papers Evaluated

The workflow could rank publications by relevance and inspect only the most promising papers. For example, it could prioritize papers based on:

* Exact variant matches
* Title or abstract relevance
* Publication type
* Recency
* The presence of functional-assay terminology

#### 4. Explore Alternative Implementations

The current paper-retrieval and experiment-extraction design may need to be reconsidered. Alternative retrieval systems, indexing strategies, batch-processing methods, or more targeted functional-evidence extraction approaches may provide better coverage and efficiency.

---

### Overall Conclusion

> Overall, the immediate cause of the false-negative results was the missing `rsID`, which prevented LitVar2 from retrieving relevant PMIDs. Although fallback publication-search methods have now been added, further optimization is necessary before they can be enabled in the main workflow without causing unacceptable processing times.
